In [3]:
!pip install pyspark

In [5]:
from pyspark import SparkContext

# Inicializando o SparkContext
sc = SparkContext("local", "AnaliseVendasRDD")

# Carregando os dados do arquivo CSV em um RDD
vendas_rdd = sc.textFile("vendas.csv")

# Transformando as linhas de texto em tuplas (id_venda, id_produto, quantidade, valor)
vendas_tuplas_rdd = vendas_rdd.map(lambda linha: tuple(linha.split(",")))

# Mapeando o RDD para (id_produto, (quantidade, valor))
produtos_rdd = vendas_tuplas_rdd.map(lambda venda: (venda[1], (int(venda[2]), float(venda[3]))))

# Calculando o total de vendas por produto (somando quantidades e valores)
total_vendas_por_produto = produtos_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Exibindo o total de vendas por produto
print("Total de vendas por produto (quantidade, valor):")
for produto in total_vendas_por_produto.collect():
    print(f"Produto ID {produto[0]}: Quantidade Total = {produto[1][0]}, Valor Total = {produto[1][1]}")

# Encontrando o produto com maior volume de vendas em termos de quantidade
produto_mais_vendido = total_vendas_por_produto.reduce(lambda a, b: a if a[1][0] > b[1][0] else b)
print(f"\nProduto mais vendido: Produto ID {produto_mais_vendido[0]} com {produto_mais_vendido[1][0]} unidades vendidas.")

# Calculando a média de vendas por produto
media_vendas_por_produto = total_vendas_por_produto.mapValues(lambda v: (v[1] / v[0]))
print("\nMédia de valor por unidade para cada produto:")
for produto in media_vendas_por_produto.collect():
    print(f"Produto ID {produto[0]}: Média por Unidade = {produto[1]:.2f}")

# Finalizando o SparkContext
sc.stop()

Total de vendas por produto (quantidade, valor):
Produto ID 101: Quantidade Total = 22, Valor Total = 220.0
Produto ID 102: Quantidade Total = 7, Valor Total = 70.0
Produto ID 103: Quantidade Total = 3, Valor Total = 30.0

Produto mais vendido: Produto ID 101 com 22 unidades vendidas.

Média de valor por unidade para cada produto:
Produto ID 101: Média por Unidade = 10.00
Produto ID 102: Média por Unidade = 10.00
Produto ID 103: Média por Unidade = 10.00
